# Assignment A1: Predict, then measure
**AIMS Edge ML for Science · Out: Day 2 · Due: start of Day 4 · ~6 hours**

In Lab 1 you measured your laptop's roofline and placed one tiny model on it. Now you **predict** how other workloads behave,
and only then check your predictions against measurements. A good prediction that turns out wrong, *with a clear account of why*,
earns more marks than a lucky one.

| Part | Mark | Weight | What |
|---|---|---|---|
| 1 | 🟥 AI-off | 40% | Three budgeting problems, on paper. Submit a photo or PDF of handwritten work. |
| 2 | 🟩 AI-on | 45% | Predict and measure a second model on your own machine (this notebook). |
| 3 | 🟥 AI-off | 15% | A one-page reflection, written by you. |

**Submit:** this notebook (executed, with outputs) + `A1_part1.pdf` + `A1_part3.pdf` (or `.md`).
Run everything on **your own** laptop, plugged in, and record its state.
You may be asked to explain any cell in a 3-minute conversation, so make sure you understand everything you submit.

## Part 1 🟥 AI-off: budgets on paper (40%)
Do these **by hand**. Show every step and state your assumptions. Where a problem asks for *your* machine's numbers,
use your Lab 1 results. Use **fp32** unless told otherwise, and 1 MAC = 2 FLOPs.

**P1. Radio-telescope RFI flagging (15%).** Assume an array of 64 dishes, each producing 4096 frequency channels at 1000 samples/s per channel.
A classifier flags radio-frequency interference on non-overlapping windows of 32 samples (one channel, one dish) using an MLP 32 → 128 → 64 → 2.
The results must be ready within 1 s.
- (a) How many windows per second? How many FLOP/s are needed in total, and how many GB/s of input data?
- (b) Write $I(B)$, the arithmetic intensity at batch size $B$ (weights read once per batch). Using **your** ridge point, what is the smallest $B$
  at which this workload is no longer memory-bound?
- (c) Can **your** laptop keep up? If not, how many laptops would it take? Use Little's law to check that the 1 s budget allows your batch size.

**P2. An LLM on a laptop (15%).** A 7-billion-parameter language model generates text one token at a time. Each token needs one pass over
**all** the weights, about 2 FLOPs per parameter. Take $\beta = 100$ GB/s and $P_\text{peak} = 1$ TFLOP/s.
- (a) At batch 1 in fp16, what is the arithmetic intensity, and the maximum tokens/s? Which roof limits it?
- (b) Repeat for 4-bit weights.
- (c) If 16 users share the model, so every pass over the weights produces 16 tokens, what changes? Up to how many users does throughput keep growing?
- (d) Your friend's laptop has 8 GB of RAM. Which of these configurations can run at all?

**P3. A wearable classifier (10%).** A device runs a 100k-MAC classifier 10 times per second, in int8. Use the 45 nm energies from Day 1:
int8 multiply 0.2 pJ, int8 add 0.03 pJ, 32-bit SRAM read 5 pJ, 32-bit DRAM read 640 pJ. Each 32-bit read brings in 4 int8 weights,
and every weight is read once per inference.
- (a) What is the energy per inference with all weights in on-chip SRAM, and with all weights in DRAM?
- (b) What average power does each case draw? How long does a 0.1 Wh battery last on inference alone?
- (c) In one sentence: what does this tell a hardware designer?



## Part 2 🟩 AI-on: predict, then measure (45%)
### 2.0 Setup and your Lab 1 tools
Paste your `measure` from Lab 1, then fill in your machine's numbers.

In [ ]:
import sys, os, time, platform
sys.path.append("..")
import numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0)

def measure(fn, n_warmup=20, n_iter=200):
    """Your Lab 1 timing helper: per-call times in microseconds."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# Your machine, from Lab 1 (re-measure if you have changed anything, e.g. plugged in).
MACHINE = dict(
    cpu="",                 # e.g. "Apple M2", "Intel i5-1135G7"
    plugged_in=True,
    P_peak=None,            # FLOP/s (from your 2048x2048 matmul)
    beta_dram=None,         # B/s   (from the largest triad)
    beta_cache=None,        # B/s   (from the mid-size triad)
    t_overhead=None,        # s     (batch-1 median latency of the jet MLP: your "overhead floor")
)
assert None not in MACHINE.values(), "fill in your Lab 1 numbers"

### 2.1 The candidates
`JetMLP` is your Lab 1 baseline. **Choose at least one** of the other three as your second model
(choosing two earns up to +5% bonus). Don't change their definitions.

In [ ]:
def mlp(sizes):
    layers = []
    for i, (a, b) in enumerate(zip(sizes[:-1], sizes[1:])):
        layers.append(nn.Linear(a, b))
        if i < len(sizes) - 2:
            layers.append(nn.ReLU())
    return nn.Sequential(*layers)

class TinyCNN(nn.Module):
    """A small CNN on 1x32x32 'jet images' (calorimeter-like). Input content is irrelevant for timing."""
    def __init__(self):
        super().__init__()
        self.f = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 5))
    def forward(self, x):
        return self.f(x)

CANDIDATES = {
    "JetMLP":  (lambda: mlp((16, 64, 32, 32, 5)),       (16,)),
    "WideMLP": (lambda: mlp((16, 512, 512, 512, 5)),    (16,)),
    "HugeMLP": (lambda: mlp((16, 4096, 4096, 5)),       (16,)),
    "TinyCNN": (lambda: TinyCNN(),                      (1, 32, 32)),
}
MY_CHOICE = ["WideMLP"]          # <- your choice(s)

### 2.2 Counting, generalised
Extend Lab 1's counting so it handles `nn.Conv2d` as well as `nn.Linear`. For a convolution, the MAC count depends on the **output** spatial size,
so you need to run a forward pass. Forward hooks are the clean way to capture that. Also compute the **activation** bytes, which matter for the CNN.

In [ ]:
def profile_counts(model, in_shape, dtype_bytes=4):
    """Returns dict(params, macs, weight_bytes, io_bytes_per_sample, act_bytes_per_sample) for ONE sample."""
    # YOUR CODE HERE
    raise NotImplementedError

counts = {}
for name, (make, shape) in CANDIDATES.items():
    counts[name] = profile_counts(make(), shape)
    c = counts[name]
    print(f"{name:8s} params {c['params']:>10,} | MACs {c['macs']:>11,} | weights {c['weight_bytes']/1e6:7.2f} MB | "
          f"I/O {c['io_bytes_per_sample']:>5} B | acts {c['act_bytes_per_sample']:>7,} B")
assert counts["JetMLP"]["macs"] == 4256

**Q2.2** 🟥 *AI-off.* Check one of your chosen model's MAC counts **by hand**, write it below, and confirm it matches.
Which candidate's weights fit in your L2 cache, which in your L3, and which in neither? (Look up your CPU's cache sizes.)

*Your answer:*

### 2.3 Predict, **before** you measure
Use a simple latency model:
$$t(B) \approx t_\text{overhead} + \max\left(\frac{\text{FLOPs}(B)}{P_\text{peak}},\; \frac{\text{bytes}(B)}{\beta}\right)$$
where bytes$(B)$ = weights + $B\times$(I/O + activations). **You** decide which $\beta$ (DRAM or cache) applies to each model, and justify it.

Fill in `predict_latency` and the β choices. Run the cell, then **do not edit it again**. The timestamp shows when you made your prediction.

In [ ]:
def predict_latency(c, B, P, beta, t0):
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
BETA_CHOICE = {          # which bandwidth ("beta_dram" or "beta_cache") applies, and WHY (one line each)
    "JetMLP":  ("?", "?"),
    "WideMLP": ("?", "?"),
    "HugeMLP": ("?", "?"),
    "TinyCNN": ("?", "?"),
}

In [ ]:
BATCHES = [1, 16, 256, 4096]
PRED = {n: [predict_latency(counts[n], B, MACHINE["P_peak"], MACHINE[BETA_CHOICE[n][0]], MACHINE["t_overhead"]) for B in BATCHES]
        for n in ["JetMLP"] + MY_CHOICE}
print("prediction locked at", time.strftime("%Y-%m-%d %H:%M:%S"))
for n, ts in PRED.items():
    print(f"{n:8s}", "  ".join(f"B={B}: {t*1e6:9.1f} µs" for B, t in zip(BATCHES, ts)))

### 2.4 Measure
Measure the median latency at each batch size, **three times** (re-running from scratch each time), and report the spread.

In [ ]:
def measure_model(name, batches=BATCHES, repeats=3):
    make, shape = CANDIDATES[name]
    model = make().eval()
    res = np.zeros((repeats, len(batches)))
    # YOUR CODE HERE
    raise NotImplementedError
    return res          # seconds, shape (repeats, len(batches))

MEAS = {n: measure_model(n) for n in ["JetMLP"] + MY_CHOICE}
for n, r in MEAS.items():
    print(f"{n:8s}", "  ".join(f"B={B}: {m*1e6:9.1f}±{s*1e6:.1f} µs" for B, m, s in zip(BATCHES, r.mean(0), r.std(0))))

### 2.5 Compare

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for n in PRED:
    l, = ax[0].loglog(BATCHES, np.array(PRED[n]) * 1e6, "--", label=f"{n} predicted")
    ax[0].errorbar(BATCHES, MEAS[n].mean(0) * 1e6, MEAS[n].std(0) * 1e6, fmt="o-", c=l.get_color(), label=f"{n} measured")
ax[0].set(xlabel="batch size", ylabel="latency per batch [µs]", title="prediction vs. measurement"); ax[0].legend(fontsize=8)

I_ax = np.logspace(-2, 4, 200)
ax[1].loglog(I_ax, np.minimum(MACHINE["P_peak"], I_ax * MACHINE["beta_dram"]) / 1e9, "k-", label="DRAM roof")
ax[1].loglog(I_ax, np.minimum(MACHINE["P_peak"], I_ax * MACHINE["beta_cache"]) / 1e9, "k:", label="cache roof")
for n in PRED:
    c = counts[n]; B = np.array(BATCHES)
    I = 2 * c["macs"] * B / (c["weight_bytes"] + B * (c["io_bytes_per_sample"] + c["act_bytes_per_sample"]))
    ax[1].plot(I, 2 * c["macs"] * B / MEAS[n].mean(0) / 1e9, "o-", label=n)
ax[1].set(xlabel="arithmetic intensity [FLOP/B]", ylabel="achieved GFLOP/s", title="my roofline"); ax[1].legend(fontsize=8)
for a in ax: a.grid(alpha=.3, which="both")
plt.tight_layout(); plt.show()

**Q2.5** For each model and batch size where prediction and measurement differ by more than 2×, give the most likely reason.
Possible culprits: overhead, the wrong roof, threading, activation traffic, the framework choosing a different kernel, cache effects.
Then design **one** small extra experiment that tests your explanation, run it below, and report what it showed.

*Your answer:*

In [ ]:
# Your extra experiment
# YOUR CODE HERE
raise NotImplementedError


## Part 3 🟥 AI-off: reflection (15%), max one page, in your own words
1. Where does each model you studied sit (compute-, memory- or overhead-bound) at B=1 and at B=4096? How confident are you, and what evidence supports it?
2. What was the biggest surprise, and what did it teach you about your machine?
3. If you had to run your second model at **1 µs** latency, which lever from Day 1 would you pull first, and why?

Write it in `A1_part3.pdf` or `.md`. We are looking for reasoning, not length.